# 10 – Interactive queries with Trino

**Goal (Task 5, parts 1–3):**
1. A Trino **cluster**: 1 coordinator + 2 workers, with the **Trino UI** → list the **nodes**; two **catalogs** (database + pile) → list the **catalogs**.
2. Queries on the **database** and on the **pile**, an **older snapshot**, and **partition pruning**.
3. A **federated query** that combines the database and the pile in one SQL statement, and which part runs **in PostgreSQL** (pushdown) and which **in Trino**.

**Trino** is a distributed SQL engine. It stores no data itself: it **queries data where it lives** (PostgreSQL, Iceberg tables in Silo, …) through **catalogs**, and can combine them in one query.

| Container | Role |
|---|---|
| `trino-coordinator` | receives SQL, plans it, splits it into tasks, collects the result, hosts the web UI (http://localhost:8080/ui/) |
| `trino-worker-1`, `trino-worker-2` | execute the tasks: read data, filter, join, aggregate |

In [1]:
import pandas as pd
from trino_pile import sql, run, explain, query_stats, wait_for_cluster, TRINO_UI

pd.set_option("display.max_colwidth", 200)
print("Trino UI:", TRINO_UI, "(log in with any username, e.g. pawhome)")

Trino UI: http://localhost:8080/ui/ (log in with any username, e.g. pawhome)


## Part 1 – The cluster

### Nodes

`system.runtime.nodes` lists every node that registered with the coordinator. The cell waits until all 3 are **active** (Trino needs ~30–60 s to start).

In [2]:
nodes = wait_for_cluster(expected_nodes=3)
nodes

,node_id,http_uri,node_version,coordinator,state
0,trino-coordinator,http://172.21.0.7:8080,476,True,active
1,trino-worker-1,http://172.21.0.10:8080,476,False,active
2,trino-worker-2,http://172.21.0.11:8080,476,False,active


- `trino-coordinator` → `coordinator = True`. With `node-scheduler.include-coordinator=false` it only plans; it does no data work.
- `trino-worker-1`, `trino-worker-2` → `coordinator = False`: they do the work.

The same list is in the **Trino UI** (http://localhost:8080/ui/): "Active workers: 2", plus every query with its stages and splits.

### Catalogs

A **catalog** = one configured data source (a file in `trino/catalog/`):

| Catalog | File | Connector | What it reads |
|---|---|---|---|
| `pawhome_db` | `pawhome_db.properties` | `postgresql` | the operational database (Task 1) |
| `pile` | `pile.properties` | `iceberg` (JDBC catalog = the same SQLite file as PyIceberg) | the Iceberg tables in Silo (Tasks 3–4) |
| `system` | built in | | information about Trino itself (nodes, queries) |

In [3]:
sql("SHOW CATALOGS")

,Catalog
0,pawhome_db
1,pile
2,system


In [4]:
display(sql("SHOW SCHEMAS FROM pawhome_db"))
display(sql("SHOW SCHEMAS FROM pile"))       # = the medallion zones
sql("SHOW TABLES FROM pile.gold")

,Schema
0,information_schema
1,pg_catalog
2,public


,Schema
0,bronze
1,gold
2,information_schema
3,silver
4,system


,Table
0,breed_popularity
1,found_home_predictions
2,shelter_monthly_activity


## Part 2 – Queries on the database and on the pile

### A query on the operational database (PostgreSQL)

In [5]:
sql("""
    SELECT species, status, count(*) AS animals
    FROM pawhome_db.public.animals
    GROUP BY species, status
    ORDER BY species, status
""")

,species,status,animals
0,bird,adopted,4
1,bird,available,76
2,bird,reserved,35
3,cat,adopted,72
4,cat,available,703
5,cat,reserved,297
6,dog,adopted,101
7,dog,available,885
8,dog,reserved,362
9,guinea pig,adopted,9


### A query on the pile (Iceberg table)

In [6]:
sql("""
    SELECT species, count(*) AS animals, round(avg(age_years), 1) AS avg_age,
           sum(CASE WHEN found_home THEN 1 ELSE 0 END) AS found_home
    FROM pile.silver.animals
    GROUP BY species
    ORDER BY animals DESC
""")

,species,animals,avg_age,found_home
0,dog,1348,7.5,463
1,cat,1072,8.2,369
2,rabbit,304,4.6,105
3,guinea pig,169,2.9,53
4,bird,115,4.0,39


### Reading an older snapshot (time travel)

Every Iceberg table has hidden **metadata tables**, like `"animals$snapshots"`. With `FOR VERSION AS OF <snapshot_id>` Trino reads the table **as it was** at that snapshot.

In [7]:
snapshots = sql("""
    SELECT snapshot_id, committed_at, operation,
           element_at(summary, 'added-records') AS added_rows, element_at(summary, 'total-records') AS total_rows
    FROM pile.silver."animals$snapshots"
    ORDER BY committed_at
""")
snapshots

,snapshot_id,committed_at,operation,added_rows,total_rows
0,468205997780947077,2026-11-27 14:41:28.662+00:00,append,3006,3006
1,153363407765370715,2026-11-27 14:41:41.761+00:00,overwrite,3005,3005
2,7282769785284418261,2026-11-27 14:41:41.864+00:00,append,1,3006
3,6838324769754947321,2026-11-27 14:41:41.957+00:00,append,2,3008


In [8]:
first_snapshot = int(snapshots.snapshot_id.iloc[0])

pd.DataFrame({
    "version": [f"OLD (snapshot {first_snapshot})", "CURRENT"],
    "animals": [sql(f"SELECT count(*) FROM pile.silver.animals FOR VERSION AS OF {first_snapshot}").iloc[0, 0],
                sql("SELECT count(*) FROM pile.silver.animals").iloc[0, 0]],
})

,version,animals
0,OLD (snapshot 468205997780947077),3006
1,CURRENT,3008


In [9]:
# Rows that are different between the old snapshot and now (new or changed animals)
sql(f"""
    SELECT id, name, species, breed FROM pile.silver.animals
    EXCEPT
    SELECT id, name, species, breed FROM pile.silver.animals FOR VERSION AS OF {first_snapshot}
    ORDER BY id
""")

,id,name,species,breed
0,1,Charlie (renamed),cat,Devon Rex
1,3008,Silver Test 1,dog,French Bulldog
2,3009,Silver Test 2,cat,Unknown


### Partition pruning

`gold.shelter_monthly_activity` is partitioned by **month** (Task 4). `SHOW CREATE TABLE` shows the partition spec:

In [10]:
print(sql("SHOW CREATE TABLE pile.gold.shelter_monthly_activity").iloc[0, 0])

CREATE TABLE pile.gold.shelter_monthly_activity (
   shelter_id bigint,
   shelter_name varchar,
   city varchar,
   month_start date,
   applications bigint,
   successful bigint,
   adoptions bigint,
   rejected bigint,
   avg_days_to_decision double,
   success_rate double,
   _built_at timestamp(6) with time zone
)
WITH (
   format = 'PARQUET',
   format_version = 2,
   location = 's3://pawhome-pile/gold/shelter_monthly_activity',
   partitioning = ARRAY['month(month_start)']
)


In [11]:
# One partition (= one month) per row, with the number of data files in it
sql("""
    SELECT partition, record_count, file_count
    FROM pile.gold."shelter_monthly_activity$partitions"
    ORDER BY partition DESC
""").head(6)

,partition,record_count,file_count
0,(month: 681),50,1
1,(month: 680),50,1
2,(month: 679),50,1
3,(month: 678),50,1
4,(month: 677),50,1
5,(month: 676),50,1


Now the **same aggregation twice**: once over the whole table, once with a **filter on the partition column** (one month). Trino's statistics show how much data each query really read.

In [12]:
last_month = sql("SELECT max(month_start) FROM pile.gold.shelter_monthly_activity").iloc[0, 0]
month = (pd.Timestamp(last_month) - pd.offsets.MonthBegin(1)).date()     # last FULL month

full_result, full_stats = run("""
    SELECT sum(applications) AS applications, sum(adoptions) AS adoptions
    FROM pile.gold.shelter_monthly_activity
""")
month_result, month_stats = run(f"""
    SELECT sum(applications) AS applications, sum(adoptions) AS adoptions
    FROM pile.gold.shelter_monthly_activity
    WHERE month_start >= DATE '{month}' AND month_start < DATE '{month}' + INTERVAL '1' MONTH
""")

pd.DataFrame({"whole table": query_stats(full_stats), f"only {month:%B %Y}": query_stats(month_stats)})

,whole table,only September 2026
rows read,998,50
bytes read,147021,6409
physical input bytes,147021,6409
splits (file pieces),34,10
elapsed ms,247,341
Trino UI,http://localhost:8080/ui/query.html?20261127_165346_00037_5u387,http://localhost:8080/ui/query.html?20261127_165346_00038_5u387


**What to notice:** with the partition filter Trino read only the files of **one month** (far fewer rows, bytes and splits). Iceberg's metadata says which month every file holds, so the other files are **skipped without opening them**: that's **partition pruning**. Open the "Trino UI" link of each query to see the same numbers.

In [13]:
# The plan: the filter on month_start is handed to the Iceberg table scan (constraint on [month_start])
explain(f"""
    SELECT sum(applications) FROM pile.gold.shelter_monthly_activity
    WHERE month_start >= DATE '{month}' AND month_start < DATE '{month}' + INTERVAL '1' MONTH
""", keep=["TableScan", "constraint", "month_start"]);

    └─ TableScan[table = pile:gold.shelter_monthly_activity$data@1969985568026517020 constraint on [month_start]]
           4:month_start:date


In [14]:
# The same idea on the second gold table, partitioned by species
_, all_species = run("SELECT sum(animals) FROM pile.gold.breed_popularity")
_, only_dogs = run("SELECT sum(animals) FROM pile.gold.breed_popularity WHERE species = 'dog'")
pd.DataFrame({"all species": query_stats(all_species), "species = 'dog'": query_stats(only_dogs)})

,all species,species = 'dog'
rows read,690,468
bytes read,33488,12261
physical input bytes,33488,12261
splits (file pieces),14,10
elapsed ms,237,137
Trino UI,http://localhost:8080/ui/query.html?20261127_165347_00040_5u387,http://localhost:8080/ui/query.html?20261127_165347_00041_5u387


## Part 3 – A federated query (database + pile in one SQL statement)

**Business question:** *Which animals are **still available right now**, have the **lowest predicted chance** of finding a home, and how many **pending applications** do they have today?* → the shelter should promote these animals first.

- the **prediction** comes from the **pile** (`pile.gold.found_home_predictions`, Task 4 machine learning)
- the **current status** and the **pending applications** come **live from the database** (`pawhome_db.public.animals`, `applications`)

In [15]:
FEDERATED = """
SELECT p.id, p.name, p.species, p.prob_found_home,
       a.status AS status_now,
       coalesce(l.pending_applications, 0) AS pending_applications_now
FROM pile.gold.found_home_predictions AS p                         -- from the PILE (Iceberg)
JOIN pawhome_db.public.animals AS a                                -- LIVE from PostgreSQL
  ON a.id = p.id
LEFT JOIN (
    SELECT animal_id, count(*) AS pending_applications             -- LIVE from PostgreSQL
    FROM pawhome_db.public.applications
    WHERE status = 'pending'
    GROUP BY animal_id
) AS l ON l.animal_id = p.id
WHERE a.status = 'available'
ORDER BY p.prob_found_home ASC
LIMIT 10
"""
sql(FEDERATED)

,id,name,species,prob_found_home,status_now,pending_applications_now
0,414,Molly,dog,0.1418,available,0
1,746,Coco,dog,0.1470,available,0
2,2446,Jack,dog,0.1513,available,0
3,1818,Finn,dog,0.1522,available,0
4,2677,Ginger,dog,0.1540,available,0
5,781,Bruno,guinea pig,0.1550,available,0
6,52,Cooper,dog,0.1560,available,0
7,1346,Gracie,guinea pig,0.1562,available,0
8,1182,Teddy,dog,0.1565,available,0
9,1912,Buddy,dog,0.1565,available,0


### What runs in PostgreSQL and what runs in Trino?

`EXPLAIN` shows the plan. Look at the **table scans** of `pawhome_db`:
- **Predicate pushdown** (filters sent to the database): `constraint on [status]` / `WHERE status = 'pending'`, so PostgreSQL returns only matching rows.
- **Projection pushdown** (only needed columns): the scan lists only the few columns used (e.g. `id`, `status`), not all 12 columns of `animals`.
- **Aggregation pushdown**: the `count(*) … GROUP BY animal_id` appears **inside** the scan as `Query[SELECT … GROUP BY …]`, so PostgreSQL counts.

Everything **above** the scans runs **in Trino** (on the workers): reading the Iceberg file of `found_home_predictions`, the **joins** between the two sources (data from two different systems can only be joined in Trino), **ORDER BY … LIMIT** (`TopN`) and the final output.

In [16]:
plan = explain(FEDERATED, keep=["TableScan", "Query[", "constraint", "pawhome_db", "pile:", "Join", "TopN", "Aggregate"])

       └─ TopN[count = 10, orderBy = [prob_found_home ASC NULLS LAST]]
    LeftJoin[criteria = (pfgnrtd = pfgnrtd_11), distribution = REPLICATED]
    ├─ TopNPartial[count = 10, orderBy = [prob_found_home ASC NULLS LAST]]
    │  └─ InnerJoin[criteria = (pfgnrtd = id), distribution = REPLICATED]
    │     ├─ ScanFilter[table = pawhome_db:Query[SELECT "status", CAST("id" AS bigint) AS "_pfgnrtd_0" FROM "pawhome"."public"."animals" WHERE "status" = ?] columns=[status:varchar(20):varchar, _pfgnrtd_0:bigint:bigint], dynamicFilters = {pfgnrtd = #df_773}]
    TableScan[table = pile:gold.found_home_predictions$data@4953086112834100446]
    TableScan[table = pawhome_db:Query[SELECT "_pfgnrtd_0", CAST("animal_id" AS bigint) AS "_pfgnrtd_1" FROM (SELECT "animal_id", count(*) AS "_pfgnrtd_0" FROM "pawhome"."public"."applications" WHERE "status" = ? GROUP BY "animal_id") o] columns=[_pfgnrtd_0:bigint:bigint, _pfgnrtd_1:bigint:bigint]]


In [17]:
print(plan)      # the full plan, if you want to see everything

Trino version: 476
Fragment 0 [SINGLE]
    Output layout: [pfgnrtd, name, species, prob_found_home, status_8, expr_7]
    Output partitioning: SINGLE []
    Output[columnNames = [id, name, species, prob_found_home, status_now, pending_applications_now]]
    │   Layout: [pfgnrtd:bigint, name:varchar, species:varchar, prob_found_home:double, status_8:varchar(20), expr_7:bigint]
    │   Estimates: {rows: 10 (1.88kB), cpu: 0, memory: 0B, network: 0B}
    │   id := pfgnrtd
    │   status_now := status_8
    │   pending_applications_now := expr_7
    └─ Project[]
       │   Layout: [pfgnrtd:bigint, name:varchar, species:varchar, prob_found_home:double, status_8:varchar(20), expr_7:bigint]
       │   Estimates: {rows: 10 (1.88kB), cpu: 1.88k, memory: 0B, network: 0B}
       │   expr_7 := COALESCE(pfgnrtd_10, bigint '0')
       └─ TopN[count = 10, orderBy = [prob_found_home ASC NULLS LAST]]
          │   Layout: [pfgnrtd:bigint, name:varchar, species:varchar, prob_found_home:double, status_8:v

In [18]:
# A second, simple example: the WHOLE aggregation is pushed down to PostgreSQL
explain("""
    SELECT status, count(*) FROM pawhome_db.public.applications GROUP BY status
""", keep=["TableScan", "Query["]);

    └─ TableScan[table = pawhome_db:public.applications pawhome.public.applications columns=[status:varchar(20):varchar]]
